# Extraction Testing Notebook

This notebook tests the complete extraction pipeline to see:
1. **What output does it give** - All extracted data
2. **How text is extracted** - Full text and chunks
3. **Section-level metadata** - Number of images, formulas, tables in each section
4. **Complete capabilities** - Everything the extraction can do

## Setup and Imports

In [1]:
import sys
from pathlib import Path
import json
from pprint import pprint

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Import extraction module
from backend.extraction.extraction import PDFExtractor

print("✓ Imports successful!")

/home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Imports successful!


In [2]:
# Initialize the PDF Extractor
extractor = PDFExtractor()

# Available PDFs to test
pdf_path = Path("../input/MemGPT.pdf")  # You can change this to test other PDFs

print(f"✓ Extractor initialized")
print(f"✓ Will test with: {pdf_path.name}")

INFO:backend.extraction.pipelines.ingest_pipeline:Ingestion pipeline initialized
INFO:backend.extraction.extraction:PDF Extractor initialized


✓ Extractor initialized
✓ Will test with: MemGPT.pdf


## Run Extraction

This will extract all information from the PDF:
- Metadata (title, abstract, sections)
- Section hierarchy
- Full text
- Statistics (formulas, tables, figures per section)

In [3]:
# Run the complete extraction
result = extractor.extract(pdf_path, output_dir="../output")

print("✓ Extraction Complete!")
print(f"\nDocument ID: {result['document_id']}")
print(f"Processing Time: {result['processing_time_seconds']:.2f}s")

INFO:backend.extraction.extraction:Starting extraction for: MemGPT.pdf
INFO:backend.extraction.extraction:Step 1/3: Ingesting PDF...
INFO:backend.extraction.pipelines.ingest_pipeline:Starting ingestion for: ../input/MemGPT.pdf
INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash 57b93c9a16f3cbf8094d66b9ca9af6d0
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered picture descriptions: ['vlm', 'api']
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered layout engines: ['docling_la

✓ Extraction Complete!

Document ID: db748533-d50a-45f8-9279-b8a405dc951f


KeyError: 'processing_time_seconds'

## 1. Basic Metadata - Title & Abstract

In [4]:
metadata = result['metadata']

print("="*80)
print("PAPER TITLE")
print("="*80)
print(f"\n{metadata['title']}\n")

print("="*80)
print("ABSTRACT")
print("="*80)
print(f"\n{metadata['abstract'][:500]}...")  # First 500 characters

PAPER TITLE

MemGPT: Towards LLMs as Operating Systems

ABSTRACT

Large language models (LLMs) have revolutionized AI, but are constrained by limited context windows, hindering their utility in tasks like extended conversations and document analysis. To enable using context beyond limited context windows, we propose virtual context management , a technique drawing inspiration from hierarchical memory systems in traditional operating systems which provide the illusion of an extended virtual memory via paging between physical memory and disk. Using this techniqu...


## 2. Sections with Stats (Formulas, Tables, Figures per Section)

**This is the key feature** - each section shows how many formulas, tables, figures, and text blocks it contains!

In [5]:
print("="*80)
print("SECTIONS WITH DETAILED STATISTICS")
print("="*80)
print()

for i, section in enumerate(metadata['sections'], 1):
    indent = "  " * (section['level'] - 1)
    print(f"{indent}{i}. {section['original_name']}")
    print(f"{indent}   Level: {section['level']} | Page: {section['page_start']}")
    
    # Show statistics for each section
    stats = section['stats']
    print(f"{indent}   📊 Stats: {stats['formulas']} formulas, {stats['tables']} tables, "
          f"{stats['figures']} figures, {stats['text_blocks']} text blocks")
    print()

SECTIONS WITH DETAILED STATISTICS

1. 1. Introduction
   Level: 1 | Page: 2


KeyError: 'stats'

## 3. Global Statistics (Document-Wide)

In [6]:
stats = metadata['global_stats']

print("="*80)
print("GLOBAL DOCUMENT STATISTICS")
print("="*80)
print()
print(f"📄 Total Pages:       {stats['total_pages']}")
print(f"📑 Total Sections:    {stats['total_sections']}")
print(f"🔢 Total Formulas:    {stats['total_formulas']}")
print(f"📊 Total Tables:      {stats['total_tables']}")
print(f"🖼️  Total Figures:     {stats['total_figures']}")
print(f"📝 Total Text Blocks: {stats['total_text_blocks']}")
print()
print(f"Max Section Depth: {stats['max_section_depth']}")

GLOBAL DOCUMENT STATISTICS

📄 Total Pages:       13
📑 Total Sections:    15
🔢 Total Formulas:    0
📊 Total Tables:      3
🖼️  Total Figures:     8


KeyError: 'total_text_blocks'

## 4. Paper Analysis (LLM Inferences)

In [7]:
inference = metadata['inference']

print("="*80)
print("PAPER ANALYSIS (LLM-BASED)")
print("="*80)
print()
print(f"📚 Paper Type:   {inference['paper_type']}")
print(f"📈 Difficulty:   {inference['difficulty']}")
print(f"🔢 Math Heavy:   {'Yes ✓' if inference['math_heavy'] else 'No ✗'}")
print()
print("💡 Reasoning:")
print(f"   {inference['reasoning']}")

PAPER ANALYSIS (LLM-BASED)

📚 Paper Type:   System
📈 Difficulty:   medium
🔢 Math Heavy:   No ✗

💡 Reasoning:


KeyError: 'reasoning'

## 5. Section Hierarchy Structure

In [8]:
hierarchy = result['hierarchy']

print("="*80)
print("SECTION HIERARCHY")
print("="*80)
print()
print(f"Total Sections:   {hierarchy['total_sections']}")
print(f"Max Depth:        {hierarchy['max_depth']}")
print(f"Confidence Score: {hierarchy['confidence_score']:.2%}")
print()
print("Hierarchy Tree:")
print()

def print_hierarchy_tree(sections, level=0):
    """Recursively print hierarchy tree."""
    for section in sections:
        indent = "  " * level
        icon = "├─" if level > 0 else "●"
        print(f"{indent}{icon} {section['title']} (Level {section['level']})")
        if section.get('children'):
            print_hierarchy_tree(section['children'], level + 1)

print_hierarchy_tree(hierarchy['root_sections'])

SECTION HIERARCHY



KeyError: 'total_sections'

## 6. Text Extraction - How is Text Extracted?

In [10]:
# Show full text preview
full_text = result['full_text']

print("="*80)
print("FULL TEXT EXTRACTION")
print("="*80)
print()
print(f"Total Characters: {len(full_text):,}")
print(f"Total Words: {len(full_text.split()):,}")
print()
print("First 1000 characters:")
print("-" * 80)
print(full_text)

print("-" * 80)

FULL TEXT EXTRACTION

Total Characters: 58,089
Total Words: 8,786

First 1000 characters:
--------------------------------------------------------------------------------
MemGPT: Towards LLMs as Operating Systems
Charles Packer 1 Sarah Wooders 1 Kevin Lin 1 Vivian Fang 1 Shishir G. Patil 1 Ion Stoica 1 Joseph E. Gonzalez 1
Abstract
Large language models (LLMs) have revolutionized AI, but are constrained by limited context windows, hindering their utility in tasks like extended conversations and document analysis. To enable using context beyond limited context windows, we propose virtual context management , a technique drawing inspiration from hierarchical memory systems in traditional operating systems which provide the illusion of an extended virtual memory via paging between physical memory and disk. Using this technique, we introduce MemGPT (MemoryGPT), a system that intelligently manages different storage tiers in order to effectively provide extended context within the LLM's limi

## 7. Extraction Quality Metrics

In [11]:
print("="*80)
print("EXTRACTION QUALITY METRICS")
print("="*80)
print()
print(f"Extraction Method:  {metadata.get('extraction_method', 'N/A')}")
print(f"Fallback Used:      {metadata.get('fallback_used', False)}")
print(f"Confidence Score:   {metadata.get('confidence_score', 0):.1%}")
print(f"Field Coverage:     {metadata.get('field_coverage', 0):.1%}")
print()

if metadata.get('missing_fields'):
    print(f"Missing Fields: {', '.join(metadata['missing_fields'])}")
else:
    print("✓ All fields extracted successfully!")

EXTRACTION QUALITY METRICS

Extraction Method:  docling+groq
Fallback Used:      False
Confidence Score:   100.0%
Field Coverage:     0.0%

✓ All fields extracted successfully!


## 8. Generated Output Files

In [12]:
print("="*80)
print("SAVED FILES")
print("="*80)
print()
print("The extraction process saved the following files:")
print()
for file_type, file_path in result['files'].items():
    print(f"  {file_type.upper():20s}: {file_path}")
print()
print("You can load these JSON files to reuse the extracted data!")

SAVED FILES

The extraction process saved the following files:

  METADATA            : ../output/db748533-d50a-45f8-9279-b8a405dc951f_metadata.json
  HIERARCHY           : ../output/db748533-d50a-45f8-9279-b8a405dc951f_hierarchy.json
  FULLTEXT            : ../output/db748533-d50a-45f8-9279-b8a405dc951f_fulltext.txt
  COMPLETE            : ../output/db748533-d50a-45f8-9279-b8a405dc951f_complete.json

You can load these JSON files to reuse the extracted data!


## 9. Deep Dive - Individual Section Element IDs

Each section tracks the specific element IDs for formulas, tables, and figures. This allows you to retrieve the actual content later!

In [ ]:
# Pick a section with formulas to examine
print("="*80)
print("SECTION ELEMENT IDs EXAMPLE")
print("="*80)
print()

# Find a section with formulas
for section in metadata['sections']:
    stats = section['stats']
    if stats['formulas'] > 0 or stats['tables'] > 0 or stats['figures'] > 0:
        print(f"Section: {section['original_name']}")
        print(f"Level {section['level']} | Page {section['page_start']}")
        print()
        
        if stats.get('formula_ids'):
            print(f"  Formula IDs ({len(stats['formula_ids'])}):")
            for fid in stats['formula_ids'][:3]:  # Show first 3
                print(f"    - {fid}")
            if len(stats['formula_ids']) > 3:
                print(f"    ... and {len(stats['formula_ids']) - 3} more")
        
        if stats.get('table_ids'):
            print(f"  Table IDs ({len(stats['table_ids'])}):")
            for tid in stats['table_ids'][:3]:
                print(f"    - {tid}")
            if len(stats['table_ids']) > 3:
                print(f"    ... and {len(stats['table_ids']) - 3} more")
        
        if stats.get('figure_ids'):
            print(f"  Figure IDs ({len(stats['figure_ids'])}):")
            for fid in stats['figure_ids'][:3]:
                print(f"    - {fid}")
            if len(stats['figure_ids']) > 3:
                print(f"    ... and {len(stats['figure_ids']) - 3} more")
        
        print()
        break  # Show just one example

## 10. Complete Data Structure (JSON Preview)

In [ ]:
# Show complete metadata structure (pretty printed)
print("="*80)
print("COMPLETE METADATA STRUCTURE (First Section Only)")
print("="*80)
print()

# Create a sample with just the first section for readability
sample_metadata = {
    'title': metadata['title'],
    'abstract': metadata['abstract'][:200] + '...',
    'sections': metadata['sections'][:1],  # Just first section
    'global_stats': metadata['global_stats'],
    'inference': metadata['inference']
}

print(json.dumps(sample_metadata, indent=2))

## Summary - What Can the Extraction Do?

### ✅ Capabilities Demonstrated:

1. **Title & Abstract Extraction** - Automatically identifies and extracts
2. **Section Detection** - Finds all sections with hierarchical levels (1-5)
3. **Per-Section Statistics** - Counts formulas, tables, figures, text blocks in EACH section
4. **Global Statistics** - Document-wide counts of all elements
5. **Paper Analysis** - LLM infers paper type, difficulty level, and math intensity
6. **Section Hierarchy** - Builds complete hierarchical tree of sections
7. **Full Text Extraction** - Complete text content with layout preservation
8. **Element Tracking** - Stores IDs for each formula, table, figure per section
9. **Quality Metrics** - Confidence scores and coverage analysis
10. **Persistent Storage** - Saves all data to JSON files for reuse

### 📊 Key Features:
- **Per-section metadata**: Know exactly which sections have formulas/tables/figures
- **Element IDs**: Can retrieve actual formula/table/figure content later
- **Hierarchical structure**: Parent-child relationships between sections
- **Quality assurance**: Confidence scores and missing field tracking

## Test with Other PDFs

Want to test with a different PDF? Change the path below and run the extraction again!

In [ ]:
# Available PDFs in input folder:
available_pdfs = [
    "../input/MemGPT.pdf",
    "../input/Gated Attention.pdf",
    "../input/sample_1.pdf",
    "../input/sample_2.pdf",
]

print("Available PDFs to test:")
for i, pdf in enumerate(available_pdfs, 1):
    print(f"  {i}. {Path(pdf).name}")

# To test another PDF, uncomment and run:
# test_pdf = Path("../input/Gated Attention.pdf")
# result = extractor.extract(test_pdf, output_dir="../output")
# print(f"✓ Extracted: {result['metadata']['title']}")